# Import libraries and seurat object  

In [31]:
library(Seurat)
library(ggplot2)
library(dplyr)
library(tibble)
library(purrr)
library(patchwork)
library(rlang)
library(ggrepel)
library(tidyr)
library(tibble)


## Set the working directory

In [32]:
setwd("/Users/valentingoupille/Internship_Val_Goupille_2025")


# Load seurat object

In [33]:
seurat_obj <- readRDS("data/anndata_objects_with_complete_metadata_filtered/GeneFull_raw_unique_mult_with_metadata_grouped_filtered_100UMI_3c_SeuratV5.rds")


# Fonction 

## Summarise Seurat object

In [34]:
summarize_seurat <- function(seurat_obj,
                             group_by = NULL,
                             output = c("both", "global", "grouped"),
                             assay = "RNA",
                             slot = "counts",
                             export = c("none", "csv", "xlsx"),
                             file_path = NULL,
                             auto_filename = TRUE) {
  output <- match.arg(output)
  export <- match.arg(export)

  # Récupérer le nom de l'objet Seurat sous forme de texte
  seurat_name <- deparse(substitute(seurat_obj))

  meta_data <- seurat_obj@meta.data
  counts_matrix <- GetAssayData(seurat_obj, assay = assay, layer = slot)

  ngene_total <- sum(rowSums(counts_matrix > 0) > 0)

  # --- GLOBAL STATS ---
  global_stats <- tibble(
    Sample_ID = "Global",
    n_cells = ncol(counts_matrix),
    n_genes_detected = ngene_total,
    gene_detection_ratio = round(1, 3),
    sum_nCount_RNA = sum(meta_data$nCount_RNA, na.rm = TRUE),
    sum_nFeature_RNA = sum(meta_data$nFeature_RNA, na.rm = TRUE),
    min_nCount_RNA = min(meta_data$nCount_RNA, na.rm = TRUE),
    q1_nCount_RNA = quantile(meta_data$nCount_RNA, 0.25, na.rm = TRUE),
    median_nCount_RNA = median(meta_data$nCount_RNA, na.rm = TRUE),
    q3_nCount_RNA = quantile(meta_data$nCount_RNA, 0.75, na.rm = TRUE),
    max_nCount_RNA = max(meta_data$nCount_RNA, na.rm = TRUE),
    sd_nCount_RNA = sd(meta_data$nCount_RNA, na.rm = TRUE),
    min_nFeature_RNA = min(meta_data$nFeature_RNA, na.rm = TRUE),
    q1_nFeature_RNA = quantile(meta_data$nFeature_RNA, 0.25, na.rm = TRUE),
    mean_nFeature_RNA = mean(meta_data$nFeature_RNA, na.rm = TRUE),
    median_nFeature_RNA = median(meta_data$nFeature_RNA, na.rm = TRUE),
    q3_nFeature_RNA = quantile(meta_data$nFeature_RNA, 0.75, na.rm = TRUE),
    max_nFeature_RNA = max(meta_data$nFeature_RNA, na.rm = TRUE),
    sd_nFeature_RNA = sd(meta_data$nFeature_RNA, na.rm = TRUE)
  )

  grouped_stats <- NULL

  if (!is.null(group_by)) {
    meta_data$cell_id <- rownames(meta_data)

    grouped_list <- meta_data %>%
      group_by(.data[[group_by]]) %>%
      group_split()

    grouped_stats <- purrr::map_dfr(grouped_list, function(group_df) {
      group_id <- unique(group_df[[group_by]])
      cells <- group_df$cell_id
      sub_counts <- counts_matrix[, cells, drop = FALSE]
      n_genes_detected <- sum(rowSums(sub_counts > 0) > 0)

      tibble(
        !!group_by := group_id,
        n_cells = nrow(group_df),
        n_genes_detected = n_genes_detected,
        gene_detection_ratio = round(n_genes_detected / ngene_total, 3),
        sum_nCount_RNA = sum(group_df$nCount_RNA, na.rm = TRUE),
        sum_nFeature_RNA = sum(group_df$nFeature_RNA, na.rm = TRUE),
        min_nCount_RNA = min(group_df$nCount_RNA, na.rm = TRUE),
        q1_nCount_RNA = quantile(group_df$nCount_RNA, 0.25, na.rm = TRUE),
        median_nCount_RNA = median(group_df$nCount_RNA, na.rm = TRUE),
        q3_nCount_RNA = quantile(group_df$nCount_RNA, 0.75, na.rm = TRUE),
        max_nCount_RNA = max(group_df$nCount_RNA, na.rm = TRUE),
        sd_nCount_RNA = sd(group_df$nCount_RNA, na.rm = TRUE),
        min_nFeature_RNA = min(group_df$nFeature_RNA, na.rm = TRUE),
        q1_nFeature_RNA = quantile(group_df$nFeature_RNA, 0.25, na.rm = TRUE),
        mean_nFeature_RNA = mean(group_df$nFeature_RNA, na.rm = TRUE),
        median_nFeature_RNA = median(group_df$nFeature_RNA, na.rm = TRUE),
        q3_nFeature_RNA = quantile(group_df$nFeature_RNA, 0.75, na.rm = TRUE),
        max_nFeature_RNA = max(group_df$nFeature_RNA, na.rm = TRUE),
        sd_nFeature_RNA = sd(group_df$nFeature_RNA, na.rm = TRUE)
      )
    })

    colnames(grouped_stats)[1] <- "Sample_ID"
  }

  summary_result <- switch(output,
    global = global_stats,
    grouped = grouped_stats,
    both = dplyr::bind_rows(global_stats, grouped_stats %||% tibble())
  )

  summary_result <- summary_result %>%
    dplyr::mutate(dplyr::across(where(is.numeric), ~ round(.x, 3)))

  # --- EXPORT ---
  if (export != "none") {
    if (is.null(file_path) && auto_filename) {
      datetime <- format(Sys.time(), "%Y%m%d_%H%M%S")
      group_part <- ifelse(is.null(group_by), "nogroup", group_by)
      ext <- ifelse(export == "csv", ".csv", ".xlsx")
      file_path <- paste0("summary_", seurat_name, "_", group_part, "_", datetime, ext)
    }

    if (is.null(file_path)) {
      stop("Please provide a valid 'file_path' when exporting the summary or enable 'auto_filename'.")
    }

    if (export == "csv") {
      readr::write_csv(summary_result, file_path)
    } else if (export == "xlsx") {
      writexl::write_xlsx(summary_result, file_path)
    }
  }

  return(summary_result)
}


## Plot fonction 

In [35]:
generate_qc_plots <- function(
    seurat_obj,
    subtitle,
    group.by = "Condition_OD_timepoint",
    split.by = NULL,
    show.points = FALSE,
    color.by = NULL,
    color.mapping = NULL) {
    if (missing(subtitle) || is.null(subtitle) || subtitle == "") {
        stop("Le paramètre 'subtitle' est obligatoire.")
    }

    pt.size.val <- ifelse(show.points, 1, 0)

    # Génération du mapping de couleurs si color.by est défini
    if (!is.null(color.by)) {
        meta_df <- seurat_obj@meta.data %>%
            dplyr::select(all_of(group.by), all_of(color.by)) %>%
            dplyr::distinct()

        # Si aucun mapping fourni, on crée une palette auto
        if (is.null(color.mapping)) {
            unique_groups <- unique(meta_df[[color.by]])
            palette_fun <- scales::hue_pal()(length(unique_groups))
            color.mapping <- setNames(palette_fun, unique_groups)
        }

        meta_df$color <- color.mapping[meta_df[[color.by]]]
        named_colors <- setNames(meta_df$color, meta_df[[group.by]])
    } else if (!is.null(color.mapping)) {
        # Si pas de color.by, on suppose que color.mapping est pour group.by
        named_colors <- color.mapping
    } else {
        named_colors <- NULL
    }

    # p1
    p1 <- VlnPlot(seurat_obj,
        features = "nFeature_RNA",
        ncol = 1,
        pt.size = pt.size.val,
        group.by = group.by,
        raster = TRUE,
        layer = "counts"
    ) +
        labs(
            title = "Number of genes per cell",
            subtitle = subtitle,
            x = group.by,
            y = "Number of genes (nFeature_RNA)"
        ) +
        scale_y_continuous(limits = c(0, NA))

    if (!is.null(named_colors)) {
        p1 <- p1 + scale_fill_manual(values = named_colors)
    }

    # p2
    p2 <- VlnPlot(seurat_obj,
        features = "nCount_RNA",
        ncol = 1,
        pt.size = pt.size.val,
        group.by = group.by,
        raster = TRUE,
        layer = "counts"
    ) +
        labs(
            title = "Number of counts per cell",
            subtitle = subtitle,
            x = group.by,
            y = "Number of counts (nCount_RNA)"
        ) +
        scale_y_continuous(limits = c(0, NA))

    if (!is.null(named_colors)) {
        p2 <- p2 + scale_fill_manual(values = named_colors)
    }

    # p3 : scatter
    p3 <- FeatureScatter(seurat_obj,
        feature1 = "nCount_RNA",
        feature2 = "nFeature_RNA",
        group.by = group.by,
        raster = TRUE,
        split.by = split.by
    )

    if (!is.null(named_colors)) {
        p3 <- p3 + scale_color_manual(values = named_colors)
    }

    # p4 : ridgeplot
    cell_counts_df <- seurat_obj@meta.data %>%
        group_by(across(all_of(group.by))) %>%
        summarise(n = n(), .groups = "drop")

    cell_counts_label <- paste0(cell_counts_df[[group.by]], " (n=", cell_counts_df$n, ")", collapse = "\n")

    p4 <- RidgePlot(seurat_obj,
        features = "nCount_RNA",
        group.by = group.by,
        log = TRUE
    ) +
        theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
        labs(
            title = paste("Distribution of UMIs per cell across conditions\n", cell_counts_label),
            subtitle = subtitle,
            x = "Number of UMIs per cell (log scale)",
            y = "Density"
        )

    if (!is.null(named_colors)) {
        p4 <- p4 + scale_fill_manual(values = named_colors)
    }

    combined_p1_p2 <- p1 / p2

    return(list(
        p1 = p1,
        p2 = p2,
        combined_p1_p2 = combined_p1_p2,
        scatter_p3 = p3,
        ridge_p4 = p4
    ))
}


## Fonction to filter

## Filter by TOP Ncount or TOP NFeature

here : add a section to explain the functio

In [36]:
select_cells_by_group <- function(seurat_obj,
                                  target_total_cells,
                                  group_by = "Condition",
                                  sort_by = "nCount_RNA",
                                  random = FALSE,
                                  seed = NULL,
                                  descending = TRUE) {
  library(dplyr)
  library(tibble)
  library(purrr)

  # Vérification des colonnes
  meta <- seurat_obj@meta.data %>% rownames_to_column("cell_barcode")
  if (!(group_by %in% colnames(meta))) stop(paste("Group column", group_by, "not found."))
  if (!(sort_by %in% colnames(meta))) stop(paste("Sort column", sort_by, "not found."))

  # Nombre de groupes et cible par groupe
  n_groups <- length(unique(meta[[group_by]]))
  target_per_group <- floor(target_total_cells / n_groups)

  cat("Target total cells:", target_total_cells, "\n")
  cat("Group column:", group_by, "-", n_groups, "groups\n")
  cat("Sort by:", sort_by, " | Descending:", descending, "\n")
  cat("Cells per group:", target_per_group, "\n")
  if (random && !is.null(seed)) cat("Random selection with seed:", seed, "\n")

  # Sélection cellule par groupe
  get_cells <- function(df_group, n_target) {
    n_available <- nrow(df_group)

    if (n_available <= n_target) {
      message("Group ", unique(df_group[[group_by]]), ": only ", n_available, " cells available — keeping all.")
      return(df_group$cell_barcode)
    } else {
      if (random) {
        if (!is.null(seed)) set.seed(seed)
        return(sample(df_group$cell_barcode, size = n_target))
      } else {
        sorted_df <- if (descending) {
          df_group %>% arrange(desc(.data[[sort_by]]))
        } else {
          df_group %>% arrange(.data[[sort_by]])
        }
        return(sorted_df %>%
          slice_head(n = n_target) %>%
          pull(cell_barcode))
      }
    }
  }

  # Application par groupe
  selected_cells <- meta %>%
    group_by(.data[[group_by]]) %>%
    group_split() %>%
    map(~ get_cells(.x, target_per_group)) %>%
    unlist()

  # Sous-échantillonnage
  seurat_filtered <- subset(seurat_obj, cells = selected_cells)

  # Résumé
  summary_stats <- seurat_filtered@meta.data %>%
    group_by(.data[[group_by]]) %>%
    summarise(n_cells = n(), .groups = "drop")

  return(list(seurat = seurat_filtered, summary = summary_stats))
}


## Filter by Types of Genes (mRNA (CDS), rRNA, tRNA, tmRNA etc.)

In [37]:
filter_seurat_by_gene_type <- function(seurat_obj, gene_types_to_keep = c("CDS"), assay = "RNA") {
  cat("=== Loading Seurat object ===\n")

  # Extract feature metadata
  cat("Extracting gene metadata...\n")
  feature_metadata <- seurat_obj[[assay]][[]] %>%
    as.data.frame() %>%
    tibble::rownames_to_column("gene_name")

  # Show gene type distribution BEFORE filtering
  cat("\n--- Gene type distribution BEFORE filtering ---\n")
  feature_metadata %>%
    count(feature) %>%
    mutate(percentage = round(n / sum(n) * 100, 2)) %>%
    arrange(desc(n)) %>%
    print()

  # Filter genes
  cat("\nFiltering genes by types:", paste(gene_types_to_keep, collapse = ", "), "\n")
  genes_to_keep <- feature_metadata %>%
    filter(feature %in% gene_types_to_keep) %>%
    pull(gene_name)

  cat("Number of genes to keep:", length(genes_to_keep), "\n")

  # Show gene type distribution AFTER filtering
  cat("\n--- Gene type distribution AFTER filtering ---\n")
  feature_metadata %>%
    filter(feature %in% gene_types_to_keep) %>%
    count(feature) %>%
    print()

  # Create filtered Seurat object
  cat("\nCreating filtered Seurat object...\n")
  seurat_filtered <- subset(seurat_obj, features = genes_to_keep)

  # Display filtering stats
  cat("\n--- Filtered Seurat object statistics ---\n")
  cat("Number of cells:", ncol(seurat_filtered), "\n")
  cat("Number of genes:", nrow(seurat_filtered), "\n")

  # Show metadata info
  cat("\n--- Metadata overview ---\n")
  cat("Dimensions:", dim(seurat_filtered@meta.data), "\n")
  cat("Columns:", paste(colnames(seurat_filtered@meta.data), collapse = ", "), "\n")
  cat("First 6 rows:\n")
  print(head(seurat_filtered@meta.data))

  # Summary of filtering
  cat("\n--- Filtering summary ---\n")
  cat("Original object - Genes:", nrow(seurat_obj), " | Cells:", ncol(seurat_obj), "\n")
  cat("Filtered object - Genes:", nrow(seurat_filtered), " | Cells:", ncol(seurat_filtered), "\n")
  cat("Genes removed:", nrow(seurat_obj) - nrow(seurat_filtered), "\n")
  cat("Reduction in gene count:", round((1 - nrow(seurat_filtered) / nrow(seurat_obj)) * 100, 2), "%\n")

  # Gene type distribution in final object
  cat("\n--- Gene type distribution in the filtered object ---\n")
  feature_metadata %>%
    filter(gene_name %in% rownames(seurat_filtered)) %>%
    count(feature) %>%
    mutate(percentage = round(n / sum(n) * 100, 2)) %>%
    print()

  cat("\nFiltering completed successfully.\n")
  return(seurat_filtered)
}


## Fonction min Cell plot

In [38]:
library(Seurat)
library(ggplot2)
library(reshape2)

plot_gene_filtering_stats <- function(seurat_obj,
                                      min_cells_vec,
                                      save_plot = FALSE,
                                      filename = "gene_filtering_plot",
                                      format = "pdf", # "pdf" or "png"
                                      width = 8,
                                      height = 6,
                                      dpi = 300) {
  # Extract counts and compute number of expressed genes (non-zero in at least one cell)
  counts <- GetAssayData(seurat_obj, layer = "counts")
  expressed_genes <- rowSums(counts > 0) > 0
  counts_expressed <- counts[expressed_genes, ]
  total_expressed_genes <- nrow(counts_expressed)

  # Initialize stats data frame
  stats_df <- data.frame(
    min_cells = min_cells_vec,
    genes_kept = NA,
    genes_removed = NA
  )

  # Compute gene filtering statistics
  for (i in seq_along(min_cells_vec)) {
    min_cells <- min_cells_vec[i]
    genes_to_keep <- rowSums(counts_expressed > 0) >= min_cells
    stats_df$genes_kept[i] <- sum(genes_to_keep)
    stats_df$genes_removed[i] <- total_expressed_genes - stats_df$genes_kept[i]
  }

  # Reshape for plotting
  df_melt <- melt(stats_df, id.vars = "min_cells", variable.name = "Type", value.name = "GeneCount")
  df_melt$Type <- factor(df_melt$Type,
    levels = c("genes_removed", "genes_kept"),
    labels = c("Removed genes", "Retained genes")
  )

  # Plot
  p <- ggplot(df_melt, aes(x = min_cells, y = GeneCount, color = Type)) +
    geom_line(size = 1) +
    geom_point(size = 2) +
    labs(
      title = paste0("Effect of min_cells filtering (", total_expressed_genes, " expressed genes)"),
      x = "Minimum number of cells (min_cells)",
      y = "Number of genes",
      color = "Gene category"
    ) +
    scale_color_manual(values = c("Removed genes" = "darkred", "Retained genes" = "steelblue")) +
    theme_minimal(base_size = 14)

  print(p)

  # Save plot if requested
  if (save_plot) {
    out_file <- paste0(filename, ".", format)
    ggsave(out_file, plot = p, width = width, height = height, dpi = dpi)
  }
}



Attachement du package : ‘reshape2’


L'objet suivant est masqué depuis ‘package:tidyr’:

    smiths




# General aspect of the Seurat Object

In [39]:
cat("Available assays:\n")
print(Assays(seurat_obj))

cat("\nAvailable layers:\n")
print(Layers(seurat_obj))

# choose the assay to work with
GetAssay(seurat_obj, assay = "RNA")


Available assays:
[1] "RNA"

Available layers:
[1] "counts"


Assay (v5) data with 5500 features for 2988 cells
First 10 features:
 recC, recB, recD, QLH64-00045, QLH64-00050, QLH64-00055, QLH64-00060,
QLH64-00070, QLH64-00075, QLH64-00080 
Layers:
 counts 

In [40]:
cat("\nGene annotation information:\n")
cat("Number of features in Seurat object:", nrow(seurat_obj), "\n")
cat("Number of cells in Seurat object:", ncol(seurat_obj), "\n")



Gene annotation information:
Number of features in Seurat object: 5500 
Number of cells in Seurat object: 2988 


In [41]:
# Display available columns
cat("\nAvailable columns in metadata of cells (or barcodes because there are not filtered for the moment)):\n")
colnames(seurat_obj@meta.data)
head(seurat_obj)



Available columns in metadata of cells (or barcodes because there are not filtered for the moment)):


[1] "orig.ident"                  "nCount_RNA"                 
 [3] "nFeature_RNA"                "barcode"                    
 [5] "Well_loc_rd1"                "Well_loc_rd2"               
 [7] "Well_loc_rd3"                "CultureMedium"              
 [9] "RepBio"                      "OD"                         
[11] "RepTech"                     "ODt"                        
[13] "OD_theoric"                  "OD_measured"                
[15] "Extracted_Sequence_rd1"      "Group"                      
[17] "n_genes"                     "RepTech_suffix"             
[19] "RepBio_suffix"               "group"                      
[21] "n_genes_by_counts"           "log1p_n_genes_by_counts"    
[23] "total_counts"                "log1p_total_counts"         
[25] "pct_counts_in_top_50_genes"  "pct_counts_in_top_100_genes"
[27] "pct_counts_in_top_200_genes" "pct_counts_in_top_500_genes"

,orig.ident,nCount_RNA,nFeature_RNA,barcode,Well_loc_rd1,Well_loc_rd2,Well_loc_rd3,CultureMedium,RepBio,OD,⋯,RepBio_suffix,group,n_genes_by_counts,log1p_n_genes_by_counts,total_counts,log1p_total_counts,pct_counts_in_top_50_genes,pct_counts_in_top_100_genes,pct_counts_in_top_200_genes,pct_counts_in_top_500_genes
,<fct>,<dbl>,<int>,<chr>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,⋯,<fct>,<fct>,<int>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
GATGAATC_AATGTTGC_CTGCTTTG,GATGAATC,1673,850,GATGAATC_AATGTTGC_CTGCTTTG,E1,G1,H11,M9F,M9F_A,M9F_A_OD1,⋯,A,M9F_sampling_t1,850,6.746412,1673,7.422971,28.81052,41.12373,56.18649,79.07950
ACTATGCA_GAGCTGAA_CTGCTTTG,ACTATGCA,1542,852,ACTATGCA_GAGCTGAA_CTGCTTTG,E1,D12,B11,M9F,M9F_A,M9F_A_OD1,⋯,A,M9F_sampling_t1,852,6.748760,1542,7.341484,27.04280,39.75357,54.86381,77.17250
TATCAGCA_GACTAGTA_CTGCTTTG,TATCAGCA,1499,828,TATCAGCA_GACTAGTA_CTGCTTTG,E1,C7,F1,M9F,M9F_A,M9F_A_OD1,⋯,A,M9F_sampling_t1,828,6.720220,1499,7.313220,27.95197,39.82655,54.63642,78.11875
ACCACTGT_ATCCTGTA_CTGCTTTG,ACCACTGT,1466,814,ACCACTGT_ATCCTGTA_CTGCTTTG,E1,C4,A5,M9F,M9F_A,M9F_A_OD1,⋯,A,M9F_sampling_t1,814,6.703188,1466,7.290975,26.60300,38.26739,53.75171,78.58117
CAAGACTA_AACAACCA_CTGCTTTG,CAAGACTA,1388,779,CAAGACTA_AACAACCA_CTGCTTTG,E1,B1,D11,M9F,M9F_A,M9F_A_OD1,⋯,A,M9F_sampling_t1,779,6.659294,1388,7.236339,27.66571,40.20173,56.12392,79.89914
TCTTCACA_AATCCGTC_CTGCTTTG,TCTTCACA,1289,724,TCTTCACA_AATCCGTC_CTGCTTTG,E1,F12,F3,M9F,M9F_A,M9F_A_OD1,⋯,A,M9F_sampling_t1,724,6.586172,1289,7.162397,29.16990,42.04810,57.95190,82.62219
CGCTGATC_AACGTGAT_CTGCTTTG,CGCTGATC,1292,745,CGCTGATC_AACGTGAT_CTGCTTTG,E1,A1,A9,M9F,M9F_A,M9F_A_OD1,⋯,A,M9F_sampling_t1,745,6.614726,1292,7.164720,27.70898,40.32508,56.65635,81.03715
AGAGTCAA_CCTCCTGA_CTGCTTTG,AGAGTCAA,1274,717,AGAGTCAA_CCTCCTGA_CTGCTTTG,E1,D3,B12,M9F,M9F_A,M9F_A_OD1,⋯,A,M9F_sampling_t1,717,6.576470,1274,7.150701,27.94349,40.50235,57.06436,82.96703
TTCACGCA_TGAAGAGA_CTGCTTTG,TTCACGCA,1306,723,TTCACGCA_TGAAGAGA_CTGCTTTG,E1,F4,F8,M9F,M9F_A,M9F_A_OD1,⋯,A,M9F_sampling_t1,723,6.584791,1306,7.175490,28.86677,41.42420,57.35069,82.92496


In [42]:
seurat_obj@meta.data %>%
    summarise(
        CultureMedium = n_distinct(CultureMedium),
        RepBio = n_distinct(RepBio),
        RepTech = n_distinct(RepTech),
        OD = n_distinct(OD),
        ODt = n_distinct(ODt),
        Group = n_distinct(Group),
        RepTech_suffix = n_distinct(RepTech_suffix),
        RepBio_suffix = n_distinct(RepBio_suffix)
    ) %>%
    pivot_longer(everything(), names_to = "Column", values_to = "Unique_Values")


Column,Unique_Values
<chr>,<int>
CultureMedium,2
RepBio,6
RepTech,45
OD,18
ODt,3
Group,6
RepTech_suffix,5
RepBio_suffix,3


Only 45 of 90 RepTech because of the filtered (it's normal !)

In [43]:
seurat_obj@meta.data %>%
    group_by(CultureMedium) %>%
    summarise(n_cells = n()) |>
    arrange(desc(n_cells))

seurat_obj@meta.data %>%
    group_by(RepBio) %>%
    summarise(n_cells = n()) |>
    arrange(desc(n_cells))

seurat_obj@meta.data %>%
    group_by(RepTech) %>%
    summarise(n_cells = n()) |>
    arrange(desc(n_cells))

seurat_obj@meta.data %>%
    group_by(OD) %>%
    summarise(n_cells = n()) |>
    arrange(desc(n_cells))

seurat_obj@meta.data %>%
    group_by(Group) %>%
    summarise(n_cells = n()) |>
    arrange(desc(n_cells))

seurat_obj@meta.data %>%
    group_by(RepTech_suffix) %>%
    summarise(n_cells = n()) |>
    arrange(desc(n_cells))

seurat_obj@meta.data %>%
    group_by(RepBio_suffix) %>%
    summarise(n_cells = n()) |>
    arrange(desc(n_cells))


CultureMedium,n_cells
<fct>,<int>
M9,1494
M9F,1494


RepBio,n_cells
<fct>,<int>
M9F_A,498
M9F_B,498
M9F_C,498
M9_A,498
M9_B,498
M9_C,498


RepTech,n_cells
<fct>,<int>
M9F_A_OD3_3,166
M9_A_OD3_3,166
M9F_A_OD2_2,147
M9_B_OD2_3,137
M9_B_OD1_5,136
M9_C_OD2_3,133
M9F_C_OD1_1,121
M9F_B_OD2_1,114
M9F_B_OD3_5,93


OD,n_cells
<fct>,<int>
M9F_A_OD1,166
M9F_A_OD2,166
M9F_A_OD3,166
M9F_B_OD1,166
M9F_B_OD2,166
M9F_B_OD3,166
M9F_C_OD1,166
M9F_C_OD2,166
M9F_C_OD3,166


Group,n_cells
<fct>,<int>
M9F_OD1,498
M9F_OD2,498
M9F_OD3,498
M9_OD1,498
M9_OD2,498
M9_OD3,498


RepTech_suffix,n_cells
<fct>,<int>
3,1056
1,649
5,434
4,431
2,418


RepBio_suffix,n_cells
<fct>,<int>
A,996
B,996
C,996


In [44]:
# Number of values in "CultureMedium" column
cat("\nNumber of unique values in 'CultureMedium' column:\n")
unique_culture_medium <- unique(seurat_obj$CultureMedium)
cat(length(unique_culture_medium), "\n")
cat("Unique culture medium:\n")
print(unique_culture_medium)

# Number of values in "Group" column
cat("\nNumber of unique values in 'Group' column:\n")
unique_groups <- unique(seurat_obj$Group)
cat(length(unique_groups), "\n")
cat("Unique groups:\n")
print(unique_groups)

# Number of values in "RepBio" column
cat("\nNumber of unique values in 'RepBio' column:\n")
unique_repbio <- unique(seurat_obj$RepBio)
cat(length(unique_repbio), "\n")
cat("Unique biological replicates:\n")
print(unique_repbio)

# Number of values in "RepTech" column
cat("\nNumber of unique values in 'RepTech' column:\n")
unique_reptech <- unique(seurat_obj$RepTech)
cat(length(unique_reptech), "\n")
cat("Unique technical replicates:\n")
print(unique_reptech)



Number of unique values in 'CultureMedium' column:
2 
Unique culture medium:
[1] M9F M9 
Levels: M9 M9F

Number of unique values in 'Group' column:
6 
Unique groups:
[1] M9F_OD1 M9F_OD2 M9F_OD3 M9_OD1  M9_OD2  M9_OD3 
Levels: M9F_OD1 M9F_OD2 M9F_OD3 M9_OD1 M9_OD2 M9_OD3

Number of unique values in 'RepBio' column:
6 
Unique biological replicates:
[1] M9F_A M9F_B M9F_C M9_A  M9_B  M9_C 
Levels: M9F_A M9F_B M9F_C M9_A M9_B M9_C

Number of unique values in 'RepTech' column:
45 
Unique technical replicates:
 [1] M9F_A_OD1_1 M9F_A_OD1_4 M9F_A_OD1_5 M9F_A_OD2_2 M9F_A_OD2_4 M9F_A_OD3_3
 [7] M9F_B_OD1_4 M9F_B_OD1_3 M9F_B_OD2_1 M9F_B_OD2_4 M9F_B_OD3_1 M9F_B_OD3_5
[13] M9F_C_OD1_1 M9F_C_OD1_2 M9F_C_OD1_3 M9F_C_OD2_5 M9F_C_OD2_3 M9F_C_OD2_2
[19] M9F_C_OD2_1 M9F_C_OD3_1 M9F_C_OD3_3 M9F_C_OD3_4 M9_A_OD1_3  M9_A_OD1_2 
[25] M9_A_OD1_4  M9_A_OD2_5  M9_A_OD2_3  M9_A_OD2_2  M9_A_OD3_3  M9_B_OD1_5 
[31] M9_B_OD1_3  M9_B_OD2_5  M9_B_OD2_3  M9_B_OD3_1  M9_B_OD3_3  M9_B_OD3_4 
[37] M9_C_OD1_4  M9_C_OD1_2 

In [46]:
# Display available columns
cat("\nAvailable columns in metadata genes (features):\n")
# see the genes info layer metadata
colnames(seurat_obj[["RNA"]]@meta.data)
# colnames(GetAssay(seurat_obj, assay = "RNA")@meta.data) # fait pareil

head(seurat_obj@assays$RNA@meta.data)



Available columns in metadata genes (features):


[1] "gene_id"                         "final_gene_name"                
 [3] "symbol_renamed"                  "Gene Type"                      
 [5] "Name"                            "Protein accession"              
 [7] "symbol_old"                      "locus_tag_old"                  
 [9] "n_cells_by_counts"               "mean_counts"                    
[11] "log1p_mean_counts"               "pct_dropout_by_counts"          
[13] "total_counts"                    "log1p_total_counts"             
[15] "n_cells"                         "gene_pass_union_OD_5pct"        
[17] "gene_pass_intersection_OD_5pct"  "gene_pass_union_OD_10pct"       
[19] "gene_pass_intersection_OD_10pct" "gene_pass_union_OD_15pct"       
[21] "gene_pass_intersection_OD_15pct"

,gene_id,final_gene_name,symbol_renamed,Gene Type,Name,Protein accession,symbol_old,locus_tag_old,n_cells_by_counts,mean_counts,⋯,pct_dropout_by_counts,total_counts,log1p_total_counts,n_cells,gene_pass_union_OD_5pct,gene_pass_intersection_OD_5pct,gene_pass_union_OD_10pct,gene_pass_intersection_OD_10pct,gene_pass_union_OD_15pct,gene_pass_intersection_OD_15pct
,<chr>,<chr>,<lgl>,<fct>,<fct>,<fct>,<fct>,<chr>,<int>,<dbl>,⋯,<dbl>,<int>,<dbl>,<int>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>
recC,QLH64-00030,recC,FALSE,protein-coding,exodeoxyribonuclease V subunit gamma,WHS54405.1,recC,QLH64_00030,154,0.05287818,⋯,94.84605,158,5.068904,154,TRUE,FALSE,TRUE,FALSE,FALSE,FALSE
recB,QLH64-00035,recB,FALSE,protein-coding,exodeoxyribonuclease V subunit beta,WHS54406.1,recB,QLH64_00035,114,0.03882195,⋯,96.18474,116,4.762174,114,TRUE,FALSE,TRUE,FALSE,FALSE,FALSE
recD,QLH64-00040,recD,FALSE,protein-coding,exodeoxyribonuclease V subunit alpha,WHS54407.1,recD,QLH64_00040,107,0.03714859,⋯,96.41901,111,4.718499,107,TRUE,FALSE,TRUE,FALSE,FALSE,FALSE
QLH64-00045,QLH64-00045,QLH64-00045,TRUE,protein-coding,YfiR family protein,WHS54408.1,NA,QLH64_00045,55,0.01874163,⋯,98.15930,56,4.043051,55,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE
QLH64-00050,QLH64-00050,QLH64-00050,TRUE,protein-coding,diguanylate cyclase,WHS54409.1,NA,QLH64_00050,45,0.01539491,⋯,98.49398,46,3.850148,45,TRUE,FALSE,FALSE,FALSE,FALSE,FALSE
QLH64-00055,QLH64-00055,QLH64-00055,TRUE,protein-coding,OmpA family protein,WHS57090.1,NA,QLH64_00055,48,0.01639893,⋯,98.39357,49,3.912023,48,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE
